<a href="https://colab.research.google.com/github/vedadapa/rag-pipeline/blob/main/02_rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#the loan default dataset from Kaggle.
!kaggle datasets download -d yasserh/loan-default-dataset

Dataset URL: https://www.kaggle.com/datasets/yasserh/loan-default-dataset
License(s): CC0-1.0
loan-default-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
# Unzip the downloaded dataset file.
!unzip loan-default-dataset.zip

Archive:  loan-default-dataset.zip
replace Loan_Default.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


In [ ]:
# Import pandas for data manipulation and numpy for numerical operations.
import pandas as pd
import numpy as np

In [ ]:
# Load the 'Loan_Default.csv' file into a pandas DataFrame.
df = pd.read_csv('Loan_Default.csv')

In [ ]:
# Displaying of the first row of the DataFrame for analyzing the features.
df.head(1)

,ID,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,24890,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,...,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0


In [ ]:
# Print the count of each unique value in the 'Status' column for the first 500 rows.
print(df['Status'].head(500).value_counts())

In [ ]:
# Print all column names in the DataFrame as a list.
print(df.columns.tolist())

In [ ]:
# Print the 'text' description of the first customer who defaulted (Status == 1).
print(df[df['Status'] == 1]['text'].iloc[0])

In [ ]:
# A function to convert each row's data into a descriptive text string.
def row_to_text(row):
    status = "defaulted" if row['Status'] == 1 else "did not default"
    return f"Customer aged {row['age']}, gender {row['Gender']}, credit score {row['Credit_Score']}, loan type {row['loan_type']}, region {row['Region']}, {status}."

# Apply the function to create a new 'text' column.
df['text'] = df.apply(row_to_text, axis=1)
# Print the first few entries of the new 'text' column.
print(df['text'].head())

0    Customer aged 25-34, gender Sex Not Available,...
1    Customer aged 55-64, gender Male, credit score...
2    Customer aged 35-44, gender Male, credit score...
3    Customer aged 45-54, gender Male, credit score...
4    Customer aged 25-34, gender Joint, credit scor...
Name: text, dtype: object


In [ ]:
# Balance the dataset by taking an equal number of defaulted and non-defaulted loan records.
defaulted = df[df['Status'] == 1].head(250)
not_defaulted = df[df['Status'] == 0].head(250)

# Concatenate and shuffle the balanced data.
balanced_df = pd.concat([defaulted, not_defaulted]).sample(frac=1, random_state=42)

# Extract the 'text' descriptions and create unique IDs for each record.
texts = balanced_df['text'].tolist()
ids = [f"doc:{i}" for i in range(len(texts))]

In [ ]:
# Initialize ChromaDB client and set up a sentence transformer embedding function.
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.Client()

ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Get or create a ChromaDB collection named 'loan_docs_fresh'.
collection_fresh = client.get_or_create_collection(
    "loan_docs_fresh",
    embedding_function=ef
)

# Add the balanced text data and their IDs to the ChromaDB collection.
collection_fresh.add(
    documents=texts,
    ids=ids
)

# Print the total number of documents in the collection.
print(collection_fresh.count())

# Verify the first 5 documents added to the collection.
print(collection_fresh.get(limit=5)['documents'])

500
['Customer aged 25-34, gender Sex Not Available, credit score 758, loan type type1, region south, defaulted.', 'Customer aged 55-64, gender Male, credit score 552, loan type type2, region North, defaulted.', 'Customer aged 35-44, gender Male, credit score 834, loan type type1, region south, did not default.', 'Customer aged 45-54, gender Male, credit score 587, loan type type1, region North, did not default.', 'Customer aged 25-34, gender Joint, credit score 602, loan type type1, region North, did not default.']


In [ ]:
# Query the ChromaDB collection for documents similar to 'defaulted loan customer'.
# Retrieve 3 results, including their original documents and calculated distances.
results = collection_fresh.query(
    query_texts=["defaulted loan customer"],
    n_results=3,
    include=["documents", "distances"]
)
# Print the retrieved documents.
print(results['documents'])
# Print the similarity distances for the retrieved documents.
print(results['distances'])

[['Customer aged >74, gender Male, credit score 737, loan type type1, region North, defaulted.', 'Customer aged 55-64, gender Male, credit score 692, loan type type1, region south, defaulted.', 'Customer aged 55-64, gender Male, credit score 762, loan type type1, region south, defaulted.']]
[[0.30689573287963867, 0.3125585913658142, 0.3152034282684326]]


In [ ]:
# LLM integration
from groq import Groq
groq_client = Groq(api_key="GROQ_API_KEY")

# Define the query for the RAG system.
query = "defaulted loan customer"

# Query ChromaDB to retrieve relevant documents based on the user's query.
results = collection_fresh.query(
    query_texts=[query],
    n_results=3
)
# Combine the retrieved documents into a single context string.
context = "\n".join(results['documents'][0])

# Create a prompt for the Groq model, incorporating both the context and the user's query.
prompt = f"Based on these loan records:\n{context}\n\nAnswer this: {query}"

# Send the prompt to the Groq model (llama-3.1-8b-instant) and get a response.
response = groq_client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}]
)
# Print the model's generated answer.
print(response.choices[0].message.content)

Based on the given data, the defaulted loan customer characteristics that can be determined are:

- Age Group: 55-64
- Gender: Male
- Credit Score: Below 737 (minimum 692, maximum 762)
- Loan Type: type1
- Region: South (all customers in the South did not default)
  (Note -  customer in the North region defaulted).
